# PROJECT-2. Анализ вакансий из HeadHunter

**Курс:** Data Scientist PRO (SkillFactory DSPR-2.0)

**Цель:** Анализ данных о вакансиях с HeadHunter с помощью SQL-запросов к PostgreSQL.

**База данных:** PostgreSQL (host: 84.201.134.129, dbname: project_sql)

**Таблицы:**
- `vacancies` (49197 записей) — вакансии
- `employers` (23501) — работодатели
- `areas` (1362) — регионы
- `industries` (294) — отрасли
- `employers_industries` (32333) — связь работодателей и отраслей

In [ ]:
import os
import psycopg2
import pandas as pd

# Пароль в коде не хранится: перед запуском задайте SF_PG_PASSWORD в окружении.
conn = psycopg2.connect(
    host=os.environ.get('SF_PG_HOST', '84.201.134.129'),
    dbname=os.environ.get('SF_PG_DB', 'project_sql'),
    user=os.environ.get('SF_PG_USER', 'skillfactory'),
    password=os.environ['SF_PG_PASSWORD']
)

def sql(query):
    return pd.read_sql(query, conn)

## Блок 3. Знакомство с данными

In [2]:
# 3.1. Количество вакансий
sql('SELECT COUNT(*) FROM vacancies')  # 49197

   count
0  49197

In [3]:
# 3.2. Количество работодателей
sql('SELECT COUNT(*) FROM employers')  # 23501

   count
0  23501

In [4]:
# 3.3. Количество регионов
sql('SELECT COUNT(*) FROM areas')  # 1362

   count
0  1362

In [5]:
# 3.4. Количество отраслей
sql('SELECT COUNT(*) FROM industries')  # 294

   count
0  294

## Блок 4. Исследование данных

In [6]:
# 4.1. Самый распространённый график
sql('''
SELECT schedule, COUNT(*) as cnt
FROM vacancies
GROUP BY schedule
ORDER BY cnt DESC
''')  # Полный день (35367)

         schedule    cnt
0   Полный день  35367

In [7]:
# 4.2. Количество вакансий с указанной зарплатой
sql('''
SELECT COUNT(*) FROM vacancies
WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL
''')  # 24073

   count
0  24073

In [8]:
# 4.3. Медиана salary_from и salary_to
sql('''
SELECT
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary_from) as median_from,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary_to) as median_to
FROM vacancies
WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL
''')  # 71065 / 110537

   median_from  median_to
0      71065.0   110537.0

In [9]:
# 4.4. Второе по популярности сочетание (график — занятость)
sql('''
SELECT schedule, employment, COUNT(*) as cnt
FROM vacancies
GROUP BY schedule, employment
ORDER BY cnt DESC
LIMIT 5
''')  # 2-е место: Удалённая работа — Полная занятость (7802)

           schedule        employment    cnt
0     Полный день  Полная занятость  35367
1  Удаленная работа  Полная занятость   7802

In [10]:
# 4.5. Самый востребованный опыт
sql('''
SELECT experience, COUNT(*) as cnt
FROM vacancies
GROUP BY experience
ORDER BY cnt DESC
''')  # От 1 года до 3 лет (24236)

             experience    cnt
0  От 1 года до 3 лет  24236

## Блок 5. Работодатели и регионы

In [11]:
# 5.1. ТОП-5 работодателей по количеству вакансий
sql('''
SELECT e.name, COUNT(v.id) as cnt
FROM vacancies v
JOIN employers e ON v.employer_id = e.id
GROUP BY e.name
ORDER BY cnt DESC
LIMIT 5
''')  # 1: Яндекс(1933), 5: Газпром нефть(331)

          name   cnt
0       Яндекс  1933
4  Газпром нефть   331

In [12]:
# 5.2. Регион с наибольшим количеством вакансий
sql('''
SELECT a.name, COUNT(v.id) as cnt
FROM vacancies v
JOIN areas a ON a.id = v.area_id
GROUP BY a.name
ORDER BY cnt DESC
LIMIT 1
''')  # Россия

     name    cnt
0  Россия  8419

In [13]:
# 5.3. Отрасль с наибольшим числом вакансий
sql('''
SELECT i.name, COUNT(DISTINCT v.id) as cnt
FROM vacancies v
JOIN employers e ON v.employer_id = e.id
JOIN employers_industries ei ON e.id = ei.employer_id
JOIN industries i ON ei.industry_id = i.id
GROUP BY i.name
ORDER BY cnt DESC
LIMIT 1
''')  # Информационные технологии...

                              name    cnt
0  Информационные технологии...  16529

In [14]:
# 5.4-5.6. Регион Россия, работодатель 2ГИС, вакансий в Москве
# 5.4: Россия — 8419 вакансий
# 5.5: 2ГИС — работодатель с наибольшим числом вакансий в Новосибирске
# 5.6: Москва — 3553 работодателя
sql('''
SELECT COUNT(DISTINCT e.id)
FROM vacancies v
JOIN employers e ON v.employer_id = e.id
JOIN areas a ON a.id = v.area_id
WHERE a.name = $$Москва$$
''')  # 3553

   count
0   3553

In [15]:
# 5.7. Яндекс в городах-миллионниках
sql('''
SELECT COALESCE(a.name, $$Total$$) as region, COUNT(v.id) as cnt
FROM vacancies v
JOIN employers e ON v.employer_id = e.id
JOIN areas a ON a.id = v.area_id
WHERE e.name = $$Яндекс$$
AND a.id IN (24,26,3,88,53,54,1,66,4,68,72,76,78,2,99,104)
GROUP BY ROLLUP(a.name)
ORDER BY cnt DESC
''')  # 17 строк, Total=485

     region  cnt
0    Total  485
1   Москва   54

## Блок 6. Вакансии для дата-сайентистов

In [16]:
# 6.1. Вакансии, связанные с данными
sql('''
SELECT COUNT(*) FROM vacancies
WHERE LOWER(name) LIKE $$%data%$$ OR LOWER(name) LIKE $$%данн%$$
''')  # 1771

   count
0   1771

In [17]:
# 6.2. Вакансии для начинающего DS (DS AND Junior)
sql('''
SELECT COUNT(*) FROM vacancies
WHERE (
    LOWER(name) LIKE $$%data scientist%$$
    OR LOWER(name) LIKE $$%data science%$$
    OR LOWER(name) LIKE $$%исследователь данных%$$
    OR (name LIKE $$%ML%$$ AND name NOT LIKE $$%HTML%$$)
    OR LOWER(name) LIKE $$%machine learning%$$
    OR LOWER(name) LIKE $$%машинн%обучен%$$
)
AND (
    LOWER(name) LIKE $$%junior%$$
    OR experience = $$Нет опыта$$
    OR employment = $$Стажировка$$
)
''')  # 51

   count
0     51

In [18]:
# 6.3. DS с навыком SQL или postgres
sql('''
SELECT COUNT(*) FROM vacancies
WHERE (
    LOWER(name) LIKE $$%data scientist%$$
    OR LOWER(name) LIKE $$%data science%$$
    OR LOWER(name) LIKE $$%исследователь данных%$$
    OR (name LIKE $$%ML%$$ AND name NOT LIKE $$%HTML%$$)
    OR LOWER(name) LIKE $$%machine learning%$$
    OR LOWER(name) LIKE $$%машинн%обучен%$$
)
AND (LOWER(key_skills) LIKE $$%sql%$$ OR LOWER(key_skills) LIKE $$%postgres%$$)
''')  # 229

   count
0    229

In [19]:
# 6.4. DS с навыком Python
sql('''
SELECT COUNT(*) FROM vacancies
WHERE (
    LOWER(name) LIKE $$%data scientist%$$
    OR LOWER(name) LIKE $$%data science%$$
    OR LOWER(name) LIKE $$%исследователь данных%$$
    OR (name LIKE $$%ML%$$ AND name NOT LIKE $$%HTML%$$)
    OR LOWER(name) LIKE $$%machine learning%$$
    OR LOWER(name) LIKE $$%машинн%обучен%$$
)
AND LOWER(key_skills) LIKE $$%python%$$
''')  # 357

   count
0    357

In [20]:
# 6.5. Среднее количество ключевых навыков в вакансиях DS
sql('''
SELECT ROUND(AVG(
    array_length(string_to_array(key_skills, chr(9)), 1)
)::numeric, 2) as avg_skills
FROM vacancies
WHERE (
    LOWER(name) LIKE $$%data scientist%$$
    OR LOWER(name) LIKE $$%data science%$$
    OR LOWER(name) LIKE $$%исследователь данных%$$
    OR (name LIKE $$%ML%$$ AND name NOT LIKE $$%HTML%$$)
    OR LOWER(name) LIKE $$%machine learning%$$
    OR LOWER(name) LIKE $$%машинн%обучен%$$
)
AND key_skills IS NOT NULL AND key_skills != $$$$
''')  # 6.41

   avg_skills
0        6.41

In [21]:
# 6.6. Средняя зарплата DS по опыту 3-6 лет
sql('''
SELECT experience, ROUND(AVG(
    (COALESCE(salary_from, salary_to) + COALESCE(salary_to, salary_from)) / 2.0
)) as avg_salary
FROM vacancies
WHERE (
    LOWER(name) LIKE $$%data scientist%$$
    OR LOWER(name) LIKE $$%data science%$$
    OR LOWER(name) LIKE $$%исследователь данных%$$
    OR (name LIKE $$%ML%$$ AND name NOT LIKE $$%HTML%$$)
    OR LOWER(name) LIKE $$%machine learning%$$
    OR LOWER(name) LIKE $$%машинн%обучен%$$
)
AND NOT (
    LOWER(name) LIKE $$%junior%$$
    OR experience = $$Нет опыта$$
    OR employment = $$Стажировка$$
)
AND (salary_from IS NOT NULL OR salary_to IS NOT NULL)
GROUP BY experience
ORDER BY avg_salary
''')  # От 3 до 6 лет: 256454

             experience  avg_salary
0  От 1 года до 3 лет      150182
1        Более 6 лет      157933
2      От 3 до 6 лет      256454

In [22]:
conn.close()